In [1]:
# ============================================================
# KAGGLE ONE-CELL
# KS TEMPORAL CROSS-OPERATOR TRANSFER TRAINING
# OPERATOR DEFINITION: beta (20 independent trajectories / beta)
# MODE: NO_MIXED
# STRICT DEMO-USAGE TRACKING: ENABLED
# ============================================================
#
# DATASET ROOT
# ------------
#   /kaggle/input/notebooks/xinyuanli913/ks-dataset
#
# Locked dataset semantics
# ------------------------
#   200 beta operators uniformly spaced in [0.070,0.085]
#   20 independent fixed-IC LF/HF trajectories per beta
#   model-facing uc/uf shape = [200,20,150,100]
#
# Physical saved-time pools
# -------------------------
#   BASE: 2.505,2.510,...,3.000  (100 physical time tokens)
#   OOD : 3.005,3.010,...,3.250  (50 physical time tokens)
#   solver dt = 0.001
#   saved spacing = 0.005 throughout all 150 tokens
#
# Fixed seed-42 A/B split stored in dataset
# -----------------------------------------
#   Group B = first 100 entries of rng.permutation(200)
#   Group A = remaining 100 entries
#
# NO_MIXED:
#   Group A -> BASE100 only
#   Group B -> BASE100 only
#
# MIXED:
#   Group A -> BASE100 + OOD50, natural uniform sampling from 150 tokens
#              (OOD fraction = 1/3)
#   Group B -> BASE100 only
#
# Episode
# -------
#   choose beta operator and one of its 20 fixed trajectories
#   M ~ Uniform{1,...,5}
#   choose M+1 distinct eligible time slots without replacement
#   demos : [x,t,uc,uf] repeated M times
#   query : [x,tq,ucq]
#   target: ufq
#
# Sequence length = 4M+3, right-pad to 23.
# Prediction is taken at the final valid query-uc row.
#
# STRICT DEMO TRACKING
# --------------------
#   demo_usage_count[operator,trajectory,time] increments ONLY AFTER a
#   gradient-training batch has completed opt.step(). Query/target appearances,
#   monitor cases, and checkpoint evaluation NEVER count as demo usage.
#   Every checkpoint embeds the usage state available at that exact epoch.
#   Therefore the BEST checkpoint contains the exact training-demo history
#   available to the selected BEST model.
#
# IMPORTANT CONTROL
# -----------------
#   NO_MIXED and MIXED use identical normalization, architecture, optimizer,
#   scheduler, monitor cases, checkpoint rule, and tracking logic.
#   Group-B OOD is NEVER used in training, normalization, monitoring, or
#   checkpoint selection.
# ============================================================

from __future__ import annotations

import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn


# ============================================================
# 0. CONFIG
# ============================================================

TRAIN_MODE = "NO_MIXED"  # exactly "NO_MIXED" or "MIXED"
assert TRAIN_MODE in {"NO_MIXED", "MIXED"}

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATASET_ROOT = Path(
    "/kaggle/input/notebooks/xinyuanli913/ks-dataset"
)

EXPECTED_DATASET_NAME = (
    "ks_mf_queryt_beta070_085_"
    "FD2LF72_SpecHF256_T3p25_"
    "B100late_OOD50_dt001_Nop200_Ntraj20_Nt150_Nr100.npz"
)

if TRAIN_MODE == "NO_MIXED":
    RUN_TAG = (
        "ks_beta20_latebase_B100_OOD50_T3p25_nomixed_"
        "Nop200_Ntraj20_A100_B100_strictDemo_"
        "large288_H6_L5_FF576_mixedM_pad23_150ep900step"
    )
else:
    RUN_TAG = (
        "ks_beta20_latebase_B100_OOD50_T3p25_mixed_"
        "Nop200_Ntraj20_A100_ID100OOD50_B100_ID100_strictDemo_"
        "large288_H6_L5_FF576_mixedM_pad23_150ep900step"
    )

SAVE_DIR = Path("/kaggle/working") / RUN_TAG
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ICL
MIN_DEMOS = 1
MAX_DEMOS = 5
MONITOR_FIXED_DEMOS = 5
MAX_CTX_ROWS = 4 * MAX_DEMOS + 3
assert MAX_CTX_ROWS == 23

# Training
BATCH_SIZE = 1000
EPOCHS = 150
STEPS_PER_EPOCH = 900
PRINT_EVERY = 5
N_MONITOR_CASES = 512
EVAL_BATCH_SIZE = 256

# Optimizer / scheduler
WEIGHT_DECAY = 1e-4
BASE_LR = 1e-4
MAX_LR = 2e-4
MIN_LR = 1e-5
WARMUP_FRAC = 0.08
GRAD_CLIP = 1.0

# Model
D_MODEL = 288
N_HEAD = 6
N_LAYERS = 5
D_FF = 576
DROPOUT = 0.10
EXPECTED_PARAMS = 3_484_036

# Expected dataset facts
N_OP_EXPECTED = 200
N_TRAJ_EXPECTED = 20
N_TIME_EXPECTED = 150
NR_EXPECTED = 100
NX_LF_EXPECTED = 72
NX_HF_EXPECTED = 256
T_EXPECTED = 3.25
SOLVER_DT_EXPECTED = 0.001
BASE_SNAPSHOT_DT_EXPECTED = 0.005
OOD_SNAPSHOT_DT_EXPECTED = 0.005
N_BASE_TIME_EXPECTED = 100
N_OOD_TIME_EXPECTED = 50
BASE_MIN_EXPECTED = 2.505
BASE_MAX_EXPECTED = 3.0
OOD_MIN_EXPECTED = 3.005
OOD_MAX_EXPECTED = 3.250

LF_SOLVER_TAG_EXPECTED = "KS_FD2_CENTERED_ETDRK4_N72_V1"
HF_SOLVER_TAG_EXPECTED = "KS_SPECTRAL_3OVER2_ETDRK4_N256_V1"

EPS = 1e-8

# Row types
TYPE_X_DEMO = 0
TYPE_T_DEMO = 1
TYPE_UC_DEMO = 2
TYPE_UF_DEMO = 3
TYPE_X_QUERY = 4
TYPE_T_QUERY = 5
TYPE_UC_QUERY = 6
N_ROW_TYPES = 7

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("=" * 130)
print(f"KS CROSS-OPERATOR TEMPORAL TRAINING -- {TRAIN_MODE}")
print("=" * 130)
print("DEVICE       :", DEVICE)
print("DATASET_ROOT :", DATASET_ROOT)
print("SAVE_DIR     :", SAVE_DIR)
print("Group split  : fixed A100/B100 with seed=42")
print("BASE100      : 2.505..3.000, spacing=0.005")
print("OOD50        : 3.005..3.250, spacing=0.005")
if TRAIN_MODE == "NO_MIXED":
    print("Train pools  : A=BASE, B=BASE")
else:
    print("Train pools  : A=BASE+OOD, B=BASE")
print("Normalization: matched BASE100-only across all 200 beta x 20 trajectories")
print("Monitor      : fixed 512 BASE100-only M=5 cases; selects BEST")
print("B-OOD usage  : NONE inside training")
print("Demo tracking: STRICT gradient-training demos only; queries/monitor excluded")
print("=" * 130)


# ============================================================
# 1. LOCATE NEW DATASET
# ============================================================

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f"Dataset root does not exist:\n{DATASET_ROOT}"
    )

matches = sorted(DATASET_ROOT.rglob(EXPECTED_DATASET_NAME))

if len(matches) == 0:
    # Conservative fallback: only accept a file whose required new-pool
    # keys and expected shapes/ranges match the new formal dataset.
    candidates = sorted(DATASET_ROOT.rglob("*.npz"))
    compatible = []
    for p in candidates:
        try:
            with np.load(p, allow_pickle=False) as f:
                required = {
                    "beta", "times", "x", "uc", "uf",
                    "base_time_idx", "base_times",
                    "ood_time_idx", "ood_times",
                    "mixed_time_idx", "mixed_times",
                    "T", "dt", "base_snapshot_dt", "ood_snapshot_dt",
                    "nx_lf", "nx_hf", "nr", "n_traj", "n_time",
                    "lf_solver_tag", "hf_solver_tag",
                    "operator_definition", "group_a_op_idx", "group_b_op_idx",
                }
                if not required.issubset(set(f.files)):
                    continue
                if tuple(f["uc"].shape) != (200, 20, 150, 100):
                    continue
                if tuple(f["uf"].shape) != (200, 20, 150, 100):
                    continue
                ts = np.asarray(f["times"], dtype=np.float64).reshape(-1)
                if len(ts) != 150 or abs(ts[0] - 2.505) > 1e-12 or abs(ts[-1] - 3.250) > 1e-12:
                    continue
                compatible.append(p)
        except Exception:
            pass
    matches = compatible

if len(matches) == 0:
    found = "\n".join(f"  {p}" for p in sorted(DATASET_ROOT.rglob("*.npz")))
    raise FileNotFoundError(
        "Could not locate the KS beta-only T3p25/B100late/OOD50/Nt150 dataset.\n"
        f"Expected exact filename:\n  {EXPECTED_DATASET_NAME}\n\n"
        "NPZ files found:\n" + (found if found else "  (none)")
    )

if len(matches) > 1:
    print("\nCompatible dataset candidates:")
    for p in matches:
        print(" ", p)
    raise RuntimeError(
        "More than one compatible new KS dataset was found. "
        "Keep only one or set an exact path in the script."
    )

DATASET_PATH = matches[0]
print("\nDATASET_PATH:", DATASET_PATH)


# ============================================================
# 2. LOAD MODEL-FACING ARRAYS + TIME POOLS
# ============================================================

with np.load(DATASET_PATH, allow_pickle=False) as f:
    operator_definition = str(np.asarray(f["operator_definition"]).reshape(-1)[0])
    beta = f["beta"][:].astype(np.float64).reshape(-1)
    times = f["times"][:].astype(np.float64).reshape(-1)
    x_grid = f["x"][:].astype(np.float32).reshape(-1)

    # Only model-facing arrays are loaded; raw native-grid arrays are not.
    uc = f["uc"][:].astype(np.float32)
    uf = f["uf"][:].astype(np.float32)

    BASE_TIME_IDX = f["base_time_idx"][:].astype(np.int64).reshape(-1)
    BASE_TIMES = f["base_times"][:].astype(np.float64).reshape(-1)
    OOD_TIME_IDX = f["ood_time_idx"][:].astype(np.int64).reshape(-1)
    OOD_TIMES = f["ood_times"][:].astype(np.float64).reshape(-1)
    MIXED_TIME_IDX = f["mixed_time_idx"][:].astype(np.int64).reshape(-1)
    MIXED_TIMES = f["mixed_times"][:].astype(np.float64).reshape(-1)

    T_saved = float(np.asarray(f["T"]).reshape(-1)[0])
    dt_saved = float(np.asarray(f["dt"]).reshape(-1)[0])
    base_snapshot_dt_saved = float(np.asarray(f["base_snapshot_dt"]).reshape(-1)[0])
    ood_snapshot_dt_saved = float(np.asarray(f["ood_snapshot_dt"]).reshape(-1)[0])
    nx_lf_saved = int(np.asarray(f["nx_lf"]).reshape(-1)[0])
    nx_hf_saved = int(np.asarray(f["nx_hf"]).reshape(-1)[0])
    nr_saved = int(np.asarray(f["nr"]).reshape(-1)[0])
    n_traj_saved = int(np.asarray(f["n_traj"]).reshape(-1)[0])
    n_time_saved = int(np.asarray(f["n_time"]).reshape(-1)[0])
    lf_solver_tag = str(np.asarray(f["lf_solver_tag"]).reshape(-1)[0])
    hf_solver_tag = str(np.asarray(f["hf_solver_tag"]).reshape(-1)[0])
    GROUP_A_OPS_STORED = f["group_a_op_idx"][:].astype(np.int64).reshape(-1)
    GROUP_B_OPS_STORED = f["group_b_op_idx"][:].astype(np.int64).reshape(-1)

    dataset_metadata = {}
    if "metadata_json" in f:
        try:
            dataset_metadata = json.loads(str(np.asarray(f["metadata_json"]).item()))
        except Exception:
            dataset_metadata = {}

N_OP, N_TRAJ, N_TIME, NX = uc.shape
assert uf.shape == uc.shape
assert beta.shape == (N_OP,)
assert times.shape == (N_TIME,)
assert x_grid.shape == (NX,)

assert N_OP == N_OP_EXPECTED
assert N_TRAJ == N_TRAJ_EXPECTED
assert N_TIME == N_TIME_EXPECTED
assert NX == NR_EXPECTED
assert nx_lf_saved == NX_LF_EXPECTED
assert nx_hf_saved == NX_HF_EXPECTED
assert nr_saved == NR_EXPECTED
assert n_traj_saved == N_TRAJ_EXPECTED
assert n_time_saved == N_TIME_EXPECTED
assert abs(T_saved - T_EXPECTED) < 1e-12
assert abs(dt_saved - SOLVER_DT_EXPECTED) < 1e-12
assert abs(base_snapshot_dt_saved - BASE_SNAPSHOT_DT_EXPECTED) < 1e-12
assert abs(ood_snapshot_dt_saved - OOD_SNAPSHOT_DT_EXPECTED) < 1e-12
assert operator_definition == "beta"
assert lf_solver_tag == LF_SOLVER_TAG_EXPECTED
assert hf_solver_tag == HF_SOLVER_TAG_EXPECTED

assert len(BASE_TIME_IDX) == N_BASE_TIME_EXPECTED
assert len(OOD_TIME_IDX) == N_OOD_TIME_EXPECTED
assert len(MIXED_TIME_IDX) == N_TIME_EXPECTED
assert abs(BASE_TIMES[0] - BASE_MIN_EXPECTED) < 1e-12
assert abs(BASE_TIMES[-1] - BASE_MAX_EXPECTED) < 1e-12
assert abs(OOD_TIMES[0] - OOD_MIN_EXPECTED) < 1e-12
assert abs(OOD_TIMES[-1] - OOD_MAX_EXPECTED) < 1e-12
assert np.array_equal(MIXED_TIME_IDX, np.arange(N_TIME, dtype=np.int64))
assert np.allclose(MIXED_TIMES, times, rtol=0.0, atol=1e-12)

# BASE = 0..99 and OOD = 100..149 on the model-facing saved-time axis.
assert np.array_equal(BASE_TIME_IDX, np.arange(0, 100, dtype=np.int64))
assert np.array_equal(OOD_TIME_IDX, np.arange(100, 150, dtype=np.int64))
assert np.array_equal(MIXED_TIME_IDX, np.arange(0, 150, dtype=np.int64))
assert np.allclose(np.diff(times), 0.005, rtol=0.0, atol=1e-12)

print("\nDataset audit:")
print("  beta        :", beta.shape, f"[{beta.min():.8f},{beta.max():.8f}]")
print("  times       :", times.shape, f"[{times[0]:.2f},{times[-1]:.2f}]")
print("  x           :", x_grid.shape)
print("  uc / uf     :", uc.shape, uf.shape)
print("  LF/HF       :", nx_lf_saved, nx_hf_saved)
print("  solver dt   :", dt_saved)
print("  BASE/OOD dt :", base_snapshot_dt_saved, ood_snapshot_dt_saved)
print("  base slots  :", len(BASE_TIME_IDX), f"{BASE_TIMES[0]:.2f}..{BASE_TIMES[-1]:.2f}")
print("  OOD slots   :", len(OOD_TIME_IDX), f"{OOD_TIMES[0]:.2f}..{OOD_TIMES[-1]:.2f}")
print("  NaN         :", bool(np.isnan(uc).any()), bool(np.isnan(uf).any()))
print("  Inf         :", bool(np.isinf(uc).any()), bool(np.isinf(uf).any()))

if np.isnan(uc).any() or np.isnan(uf).any() or np.isinf(uc).any() or np.isinf(uf).any():
    raise RuntimeError("Non-finite state values in dataset.")


# ============================================================
# 3. FIXED CROSS-OPERATOR A/B SPLIT
# ============================================================

GROUP_A_OPS = np.sort(GROUP_A_OPS_STORED).astype(np.int64)
GROUP_B_OPS = np.sort(GROUP_B_OPS_STORED).astype(np.int64)

assert GROUP_A_OPS.shape == (100,)
assert GROUP_B_OPS.shape == (100,)
assert np.intersect1d(GROUP_A_OPS, GROUP_B_OPS).size == 0
assert np.array_equal(
    np.sort(np.concatenate([GROUP_A_OPS, GROUP_B_OPS])),
    np.arange(N_OP, dtype=np.int64),
)

# Independent audit against the locked seed-42 rule.
rng_ops_check = np.random.default_rng(SEED)
perm_ops_check = rng_ops_check.permutation(N_OP)
expected_group_b = np.sort(perm_ops_check[:100]).astype(np.int64)
expected_group_a = np.sort(perm_ops_check[100:]).astype(np.int64)
assert np.array_equal(GROUP_A_OPS, expected_group_a)
assert np.array_equal(GROUP_B_OPS, expected_group_b)

is_group_a = np.zeros(N_OP, dtype=bool)
is_group_a[GROUP_A_OPS] = True

print("\nCross-operator split (dataset + seed-42 audit passed):")
print("  Group A (100):", GROUP_A_OPS)
print("  Group B (100):", GROUP_B_OPS)

np.savez(
    SAVE_DIR / "crossop_group_split.npz",
    seed=np.asarray(SEED, dtype=np.int64),
    group_a_ops=GROUP_A_OPS,
    group_b_ops=GROUP_B_OPS,
)


# ============================================================
# 4. MATCHED BASE-ONLY NORMALIZATION
# ============================================================
# Use all 200 operators and all 20 trajectories, but BASE times only.
# BASE is contiguous [:100], so slicing returns a view and avoids a giant
# advanced-indexing copy.

BASE_END_EXCLUSIVE = int(BASE_TIME_IDX[-1]) + 1
assert BASE_END_EXCLUSIVE == 100

uc_base_view = uc[:, :, :BASE_END_EXCLUSIVE, :]
uf_base_view = uf[:, :, :BASE_END_EXCLUSIVE, :]

# Accumulate in float64 for stable statistics without duplicating the arrays.
def mean_std_float64(arr: np.ndarray):
    n = arr.size
    s1 = np.sum(arr, dtype=np.float64)
    mean = s1 / n
    s2 = np.sum(np.square(arr, dtype=np.float64), dtype=np.float64)
    var = max(s2 / n - mean * mean, 0.0)
    std = math.sqrt(var) + 1e-6
    return float(mean), float(std)

uc_mean, uc_std = mean_std_float64(uc_base_view)
uf_mean, uf_std = mean_std_float64(uf_base_view)

x_mean = float(np.mean(x_grid, dtype=np.float64))
x_std = float(np.std(x_grid.astype(np.float64)) + 1e-6)
t_mean = float(np.mean(BASE_TIMES, dtype=np.float64))
t_std = float(np.std(BASE_TIMES, dtype=np.float64) + 1e-6)

x_n = ((x_grid - x_mean) / x_std).astype(np.float32)
time_scalar_n = ((times - t_mean) / t_std).astype(np.float32)
time_rows_n = np.broadcast_to(time_scalar_n[:, None], (N_TIME, NX)).copy()

print("\nMatched BASE-only normalization:")
print(f"  x  mean/std : {x_mean:.9f} / {x_std:.9f}")
print(f"  t  mean/std : {t_mean:.9f} / {t_std:.9f}")
print(f"  uc mean/std : {uc_mean:.9f} / {uc_std:.9f}")
print(f"  uf mean/std : {uf_mean:.9f} / {uf_std:.9f}")

norm_payload = {
    "x_mean": x_mean,
    "x_std": x_std,
    "t_mean": t_mean,
    "t_std": t_std,
    "uc_mean": uc_mean,
    "uc_std": uc_std,
    "uf_mean": uf_mean,
    "uf_std": uf_std,
}


# ============================================================
# 5. METRICS / RAW BASELINES
# ============================================================

def rel_l2_rows(pred, target):
    pred = np.asarray(pred, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    num = np.linalg.norm(pred - target, axis=-1)
    den = np.maximum(np.linalg.norm(target, axis=-1), EPS)
    return num / den


def summarize_pct(x):
    a = 100.0 * np.asarray(x, dtype=np.float64).reshape(-1)
    return {
        "mean": float(np.mean(a)),
        "std": float(np.std(a)),
        "median": float(np.median(a)),
        "p90": float(np.percentile(a, 90)),
        "p95": float(np.percentile(a, 95)),
        "min": float(np.min(a)),
        "max": float(np.max(a)),
    }


def summary_str(s):
    return (
        f"mean={s['mean']:.4f}% | median={s['median']:.4f}% | "
        f"std={s['std']:.4f}% | p90={s['p90']:.4f}% | max={s['max']:.4f}%"
    )

# Do not materialize a huge flattened copy; process time blocks.
def raw_pool_summary(op_pool, time_idx):
    vals = []
    for ti in np.asarray(time_idx, dtype=np.int64):
        r = rel_l2_rows(
            uc[op_pool, :, ti, :],
            uf[op_pool, :, ti, :],
        ).reshape(-1)
        vals.append(r)
    return summarize_pct(np.concatenate(vals))

print("\nRaw LF->HF discrepancy:")
print("  ALL BASE :", summary_str(raw_pool_summary(np.arange(N_OP), BASE_TIME_IDX)))
print("  A BASE   :", summary_str(raw_pool_summary(GROUP_A_OPS, BASE_TIME_IDX)))
print("  B BASE   :", summary_str(raw_pool_summary(GROUP_B_OPS, BASE_TIME_IDX)))
print("  A OOD    :", summary_str(raw_pool_summary(GROUP_A_OPS, OOD_TIME_IDX)))
print("  B OOD    :", summary_str(raw_pool_summary(GROUP_B_OPS, OOD_TIME_IDX)))


# ============================================================
# 6. ROW TYPE TEMPLATES
# ============================================================

def row_types_for_m(m):
    out = []
    for _ in range(m):
        out.extend([
            TYPE_X_DEMO,
            TYPE_T_DEMO,
            TYPE_UC_DEMO,
            TYPE_UF_DEMO,
        ])
    out.extend([
        TYPE_X_QUERY,
        TYPE_T_QUERY,
        TYPE_UC_QUERY,
    ])
    return np.asarray(out, dtype=np.int64)

ROW_TYPES_BY_M = {
    m: row_types_for_m(m)
    for m in range(MIN_DEMOS, MAX_DEMOS + 1)
}


def sample_distinct_from_pool(rng, pool, batch_size, n_pick):
    pool = np.asarray(pool, dtype=np.int64)
    n_pool = len(pool)
    if n_pick > n_pool:
        raise ValueError("n_pick exceeds pool size")

    local = rng.integers(
        0, n_pool,
        size=(batch_size, n_pick),
        dtype=np.int64,
    )

    while True:
        s = np.sort(local, axis=1)
        bad = np.any(s[:, 1:] == s[:, :-1], axis=1)
        if not np.any(bad):
            break
        nb = int(np.sum(bad))
        local[bad] = rng.integers(
            0, n_pool,
            size=(nb, n_pick),
            dtype=np.int64,
        )

    return pool[local]


# ============================================================
# 7. TRAIN BATCH -- ONLY MODE-DEPENDENT DIFFERENCE IS A TIME POOL
# ============================================================

rng_train = np.random.default_rng(SEED + 100)
ALL_OPS = np.arange(N_OP, dtype=np.int64)
OOD_START_IDX = int(OOD_TIME_IDX[0])  # 100

# STRICT tracking state. Shape follows exact physical state identity:
# [beta-operator, fixed trajectory/IC, saved time token].
# Only gradient-training DEMO appearances are counted; query/monitor appearances are excluded.
demo_usage_count = np.zeros((N_OP, N_TRAJ, N_TIME), dtype=np.int64)


def demo_coverage_summary(usage):
    usage = np.asarray(usage, dtype=np.int64)
    base_used = usage[:, :, BASE_TIME_IDX] > 0
    ood_used = usage[:, :, OOD_TIME_IDX] > 0
    gb_base_slots = np.count_nonzero(base_used[GROUP_B_OPS], axis=2)
    ga_base_slots = np.count_nonzero(base_used[GROUP_A_OPS], axis=2)
    return {
        "total_demo_uses": int(np.sum(usage)),
        "group_A_base_demo_uses": int(np.sum(usage[GROUP_A_OPS][:, :, BASE_TIME_IDX])),
        "group_A_ood_demo_uses": int(np.sum(usage[GROUP_A_OPS][:, :, OOD_TIME_IDX])),
        "group_B_base_demo_uses": int(np.sum(usage[GROUP_B_OPS][:, :, BASE_TIME_IDX])),
        "group_B_ood_demo_uses": int(np.sum(usage[GROUP_B_OPS][:, :, OOD_TIME_IDX])),
        "group_B_used_base_slots_per_trajectory_min": int(gb_base_slots.min()),
        "group_B_used_base_slots_per_trajectory_mean": float(gb_base_slots.mean()),
        "group_B_used_base_slots_per_trajectory_max": int(gb_base_slots.max()),
        "group_A_used_base_slots_per_trajectory_min": int(ga_base_slots.min()),
        "group_A_used_base_slots_per_trajectory_mean": float(ga_base_slots.mean()),
        "group_A_used_base_slots_per_trajectory_max": int(ga_base_slots.max()),
        "group_A_used_ood_slots": int(np.count_nonzero(ood_used[GROUP_A_OPS])),
        "group_B_used_ood_slots": int(np.count_nonzero(ood_used[GROUP_B_OPS])),
    }


def make_train_batch(batch_size):
    m_arr = rng_train.integers(
        MIN_DEMOS,
        MAX_DEMOS + 1,
        size=batch_size,
        dtype=np.int64,
    )

    op_batch = rng_train.integers(
        0,
        N_OP,
        size=batch_size,
        dtype=np.int64,
    )

    traj_batch = rng_train.integers(
        0,
        N_TRAJ,
        size=batch_size,
        dtype=np.int64,
    )

    mat_ctx = np.zeros((batch_size, MAX_CTX_ROWS, NX), dtype=np.float32)
    row_types = np.zeros((batch_size, MAX_CTX_ROWS), dtype=np.int64)
    padding_mask = np.ones((batch_size, MAX_CTX_ROWS), dtype=np.bool_)
    query_pos = np.empty(batch_size, dtype=np.int64)
    target_n = np.empty((batch_size, NX), dtype=np.float32)

    audit = {
        "A_episodes": 0,
        "B_episodes": 0,
        "A_slots": 0,
        "A_ood_slots": 0,
        "A_ood_queries": 0,
        "B_ood_slots": 0,
        "B_ood_queries": 0,
    }

    # Batch-local exact demo identities. These are committed to global usage
    # ONLY after opt.step() succeeds in the training loop.
    track_ops_parts = []
    track_traj_parts = []
    track_time_parts = []

    # Group by M and A/B membership so each subgroup can use one pool.
    for m in range(MIN_DEMOS, MAX_DEMOS + 1):
        for group_name in ("A", "B"):
            if group_name == "A":
                mask = (m_arr == m) & is_group_a[op_batch]
            else:
                mask = (m_arr == m) & (~is_group_a[op_batch])

            idx = np.flatnonzero(mask)
            Bm = int(idx.size)
            if Bm == 0:
                continue

            ops_m = op_batch[idx]
            traj_m = traj_batch[idx]
            S = 4 * m + 3

            if group_name == "A" and TRAIN_MODE == "MIXED":
                pool = MIXED_TIME_IDX
            else:
                pool = BASE_TIME_IDX

            slots = sample_distinct_from_pool(
                rng_train,
                pool,
                batch_size=Bm,
                n_pick=m + 1,
            )

            demo_times = slots[:, :m]
            query_times = slots[:, m]

            # EXACT demo rows only; the query time is intentionally excluded.
            track_ops_parts.append(np.repeat(ops_m, m))
            track_traj_parts.append(np.repeat(traj_m, m))
            track_time_parts.append(demo_times.reshape(-1))

            row_types[idx, :S] = ROW_TYPES_BY_M[m][None, :]
            padding_mask[idx, :S] = False

            row = 0
            for j in range(m):
                mat_ctx[idx, row, :] = x_n[None, :]
                row += 1

                mat_ctx[idx, row, :] = time_rows_n[demo_times[:, j], :]
                row += 1

                mat_ctx[idx, row, :] = (
                    (uc[ops_m, traj_m, demo_times[:, j], :] - uc_mean)
                    / uc_std
                ).astype(np.float32)
                row += 1

                mat_ctx[idx, row, :] = (
                    (uf[ops_m, traj_m, demo_times[:, j], :] - uf_mean)
                    / uf_std
                ).astype(np.float32)
                row += 1

            mat_ctx[idx, row, :] = x_n[None, :]
            row += 1

            mat_ctx[idx, row, :] = time_rows_n[query_times, :]
            row += 1

            mat_ctx[idx, row, :] = (
                (uc[ops_m, traj_m, query_times, :] - uc_mean)
                / uc_std
            ).astype(np.float32)
            row += 1

            assert row == S
            query_pos[idx] = S - 1
            target_n[idx] = (
                (uf[ops_m, traj_m, query_times, :] - uf_mean)
                / uf_std
            ).astype(np.float32)

            is_ood_slot = slots >= OOD_START_IDX
            is_ood_query = query_times >= OOD_START_IDX

            if group_name == "A":
                audit["A_episodes"] += Bm
                audit["A_slots"] += int(slots.size)
                audit["A_ood_slots"] += int(np.sum(is_ood_slot))
                audit["A_ood_queries"] += int(np.sum(is_ood_query))
            else:
                audit["B_episodes"] += Bm
                audit["B_ood_slots"] += int(np.sum(is_ood_slot))
                audit["B_ood_queries"] += int(np.sum(is_ood_query))

    # Hard protocol guards.
    if audit["B_ood_slots"] != 0 or audit["B_ood_queries"] != 0:
        raise RuntimeError("PROTOCOL VIOLATION: Group B sampled OOD during training")
    if TRAIN_MODE == "NO_MIXED" and audit["A_ood_slots"] != 0:
        raise RuntimeError("PROTOCOL VIOLATION: NO_MIXED Group A sampled OOD")

    assert np.all(
        ~padding_mask[np.arange(batch_size), query_pos]
    )

    track_ops = np.concatenate(track_ops_parts).astype(np.int64, copy=False)
    track_traj = np.concatenate(track_traj_parts).astype(np.int64, copy=False)
    track_time = np.concatenate(track_time_parts).astype(np.int64, copy=False)
    assert track_ops.shape == track_traj.shape == track_time.shape
    assert track_ops.size == int(np.sum(m_arr))

    return (
        m_arr,
        torch.from_numpy(mat_ctx).to(DEVICE, non_blocking=True),
        torch.from_numpy(row_types).to(DEVICE, non_blocking=True),
        torch.from_numpy(padding_mask).to(DEVICE, non_blocking=True),
        torch.from_numpy(query_pos).to(DEVICE, non_blocking=True),
        torch.from_numpy(target_n).to(DEVICE, non_blocking=True),
        audit,
        track_ops,
        track_traj,
        track_time,
    )


# ============================================================
# 8. FIXED BASE-ONLY MONITOR -- IDENTICAL BETWEEN MODES
# ============================================================

def make_monitor_cases(n_cases, seed):
    r = np.random.default_rng(seed)
    cases = []
    for case_id in range(n_cases):
        op = int(r.integers(0, N_OP))
        traj = int(r.integers(0, N_TRAJ))
        slots = r.choice(
            BASE_TIME_IDX,
            size=MONITOR_FIXED_DEMOS + 1,
            replace=False,
        ).astype(np.int64)
        cases.append((
            case_id,
            op,
            traj,
            slots[:MONITOR_FIXED_DEMOS],
            int(slots[-1]),
        ))
    return cases

MONITOR_CASES = make_monitor_cases(
    N_MONITOR_CASES,
    seed=SEED + 201,
)

# Save exact monitor cases so the two runs can be audited externally.
monitor_rows = []
for case_id, op, traj, demos, qti in MONITOR_CASES:
    monitor_rows.append({
        "case_id": case_id,
        "operator": op,
        "group": "A" if is_group_a[op] else "B",
        "beta": float(beta[op]),
        "trajectory": traj,
        "demo_time_idx": ",".join(str(int(v)) for v in demos),
        "demo_times": ",".join(f"{times[int(v)]:.2f}" for v in demos),
        "query_time_idx": qti,
        "query_time": float(times[qti]),
    })
pd.DataFrame(monitor_rows).to_csv(
    SAVE_DIR / "fixed_base_monitor_cases.csv",
    index=False,
)


# ============================================================
# 9. MODEL
# ============================================================

class RowSequenceTransformer(nn.Module):
    def __init__(
        self,
        row_width,
        max_ctx_rows=23,
        n_row_types=7,
        d_model=288,
        nhead=6,
        num_layers=5,
        dim_ff=576,
        dropout=0.10,
    ):
        super().__init__()

        self.row_in = nn.Linear(row_width, d_model)
        self.row_type_emb = nn.Embedding(n_row_types, d_model)
        self.pos_emb = nn.Embedding(max_ctx_rows, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            activation=torch.tanh,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            enc_layer,
            num_layers=num_layers,
        )

        self.row_out = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.Tanh(),
            nn.Linear(d_model, row_width),
        )

        self.register_buffer(
            "pos_ids",
            torch.arange(max_ctx_rows, dtype=torch.long),
            persistent=False,
        )

    def forward(self, mat_ctx, row_type_arr, key_padding_mask=None):
        B, S, _ = mat_ctx.shape
        if S > MAX_CTX_ROWS:
            raise ValueError(f"S={S} exceeds MAX_CTX_ROWS={MAX_CTX_ROWS}")
        h = self.row_in(mat_ctx)
        h = (
            h
            + self.row_type_emb(row_type_arr)
            + self.pos_emb(self.pos_ids[:S])[None, :, :]
        )
        h = self.encoder(
            h,
            src_key_padding_mask=key_padding_mask,
        )
        return self.row_out(h)


model = RowSequenceTransformer(
    row_width=NX,
    max_ctx_rows=MAX_CTX_ROWS,
    n_row_types=N_ROW_TYPES,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=N_LAYERS,
    dim_ff=D_FF,
    dropout=DROPOUT,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("\nModel:")
print("  d_model / heads / layers / FF / dropout:", D_MODEL, N_HEAD, N_LAYERS, D_FF, DROPOUT)
print("  trainable parameters:", f"{n_params:,}")
assert n_params == EXPECTED_PARAMS, (n_params, EXPECTED_PARAMS)


# ============================================================
# 10. PADDING MASK SANITY CHECK
# ============================================================

@torch.no_grad()
def padding_sanity_check():
    model.eval()
    r = np.random.default_rng(SEED + 9999)
    m = 2
    S = 4 * m + 3
    op = 0
    traj = 0
    slots = r.choice(BASE_TIME_IDX, size=m + 1, replace=False)
    demos, qti = slots[:m], int(slots[-1])

    ctx = np.zeros((1, MAX_CTX_ROWS, NX), dtype=np.float32)
    typ = np.zeros((1, MAX_CTX_ROWS), dtype=np.int64)
    mask = np.ones((1, MAX_CTX_ROWS), dtype=np.bool_)

    row = 0
    for ti in demos:
        ctx[0, row] = x_n; typ[0, row] = TYPE_X_DEMO; row += 1
        ctx[0, row] = time_rows_n[ti]; typ[0, row] = TYPE_T_DEMO; row += 1
        ctx[0, row] = ((uc[op, traj, ti] - uc_mean) / uc_std).astype(np.float32); typ[0, row] = TYPE_UC_DEMO; row += 1
        ctx[0, row] = ((uf[op, traj, ti] - uf_mean) / uf_std).astype(np.float32); typ[0, row] = TYPE_UF_DEMO; row += 1
    ctx[0, row] = x_n; typ[0, row] = TYPE_X_QUERY; row += 1
    ctx[0, row] = time_rows_n[qti]; typ[0, row] = TYPE_T_QUERY; row += 1
    ctx[0, row] = ((uc[op, traj, qti] - uc_mean) / uc_std).astype(np.float32); typ[0, row] = TYPE_UC_QUERY; row += 1
    assert row == S
    mask[0, :S] = False

    ctx2 = ctx.copy()
    typ2 = typ.copy()
    ctx2[0, S:] = r.normal(50, 20, size=(MAX_CTX_ROWS - S, NX)).astype(np.float32)
    typ2[0, S:] = r.integers(0, N_ROW_TYPES, size=MAX_CTX_ROWS - S)

    mask_t = torch.from_numpy(mask).to(DEVICE)
    out1 = model(
        torch.from_numpy(ctx).to(DEVICE),
        torch.from_numpy(typ).to(DEVICE),
        key_padding_mask=mask_t,
    )[0, S - 1]
    out2 = model(
        torch.from_numpy(ctx2).to(DEVICE),
        torch.from_numpy(typ2).to(DEVICE),
        key_padding_mask=mask_t,
    )[0, S - 1]

    d = float(torch.max(torch.abs(out1 - out2)).cpu())
    print("\nPadding sanity max diff:", f"{d:.3e}")
    if d > 1e-5:
        raise RuntimeError("Padding mask sanity check failed")
    model.train()

padding_sanity_check()


# ============================================================
# 11. OPTIMIZER / SCHEDULER
# ============================================================

TOTAL_STEPS = EPOCHS * STEPS_PER_EPOCH
WARMUP_STEPS = max(1, int(WARMUP_FRAC * TOTAL_STEPS))

opt = torch.optim.AdamW(
    model.parameters(),
    lr=MAX_LR,
    weight_decay=WEIGHT_DECAY,
)

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    opt,
    start_factor=BASE_LR / MAX_LR,
    end_factor=1.0,
    total_iters=WARMUP_STEPS,
)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt,
    T_max=max(1, TOTAL_STEPS - WARMUP_STEPS),
    eta_min=MIN_LR,
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    opt,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[WARMUP_STEPS],
)

print("\nOptimizer:")
print("  total steps  :", TOTAL_STEPS)
print("  warmup steps :", WARMUP_STEPS)
print("  LR           :", BASE_LR, "->", MAX_LR, "->", MIN_LR)


# ============================================================
# 12. FIXED BASE MONITOR EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_monitor():
    model.eval()

    pred_list = []
    target_list = []
    raw_list = []

    M = MONITOR_FIXED_DEMOS
    S = 4 * M + 3
    assert S == 23

    for start in range(0, len(MONITOR_CASES), EVAL_BATCH_SIZE):
        subset = MONITOR_CASES[start:start + EVAL_BATCH_SIZE]
        B = len(subset)

        ctx = np.empty((B, S, NX), dtype=np.float32)
        typ = np.empty((B, S), dtype=np.int64)
        target = np.empty((B, NX), dtype=np.float32)
        raw = np.empty((B, NX), dtype=np.float32)

        for i, (_, op, traj, demos, qti) in enumerate(subset):
            row = 0
            for ti in demos:
                ctx[i, row] = x_n; typ[i, row] = TYPE_X_DEMO; row += 1
                ctx[i, row] = time_rows_n[ti]; typ[i, row] = TYPE_T_DEMO; row += 1
                ctx[i, row] = ((uc[op, traj, ti] - uc_mean) / uc_std).astype(np.float32); typ[i, row] = TYPE_UC_DEMO; row += 1
                ctx[i, row] = ((uf[op, traj, ti] - uf_mean) / uf_std).astype(np.float32); typ[i, row] = TYPE_UF_DEMO; row += 1
            ctx[i, row] = x_n; typ[i, row] = TYPE_X_QUERY; row += 1
            ctx[i, row] = time_rows_n[qti]; typ[i, row] = TYPE_T_QUERY; row += 1
            ctx[i, row] = ((uc[op, traj, qti] - uc_mean) / uc_std).astype(np.float32); typ[i, row] = TYPE_UC_QUERY; row += 1
            assert row == S
            target[i] = uf[op, traj, qti]
            raw[i] = uc[op, traj, qti]

        out_n = model(
            torch.from_numpy(ctx).to(DEVICE, non_blocking=True),
            torch.from_numpy(typ).to(DEVICE, non_blocking=True),
            key_padding_mask=None,
        )[:, S - 1, :]

        pred = out_n.detach().cpu().numpy().astype(np.float32) * uf_std + uf_mean
        pred_list.append(pred)
        target_list.append(target)
        raw_list.append(raw)

    pred = np.concatenate(pred_list)
    target = np.concatenate(target_list)
    raw = np.concatenate(raw_list)

    pred_rel = rel_l2_rows(pred, target)
    raw_rel = rel_l2_rows(raw, target)

    model.train()
    return {
        "pred": summarize_pct(pred_rel),
        "raw": summarize_pct(raw_rel),
    }


# ============================================================
# 13. CHECKPOINT HELPERS / AUTO-RESUME
# ============================================================

LATEST_CKPT = SAVE_DIR / f"{RUN_TAG}_latest_fullstate.pt"
BEST_CKPT = SAVE_DIR / f"{RUN_TAG}_best_trainmonitor_fullstate.pt"
FINAL_CKPT = SAVE_DIR / f"{RUN_TAG}_final_fullstate.pt"
HISTORY_JSON = SAVE_DIR / "history.json"
HISTORY_CSV = SAVE_DIR / "history.csv"
BEST_DEMO_USAGE_NPZ = SAVE_DIR / f"{RUN_TAG}_best_demo_usage.npz"
FINAL_DEMO_USAGE_NPZ = SAVE_DIR / f"{RUN_TAG}_final_demo_usage.npz"
DEMO_USAGE_AUDIT_JSON = SAVE_DIR / f"{RUN_TAG}_demo_usage_audit.json"


def get_rng_state():
    state = {
        "python": random.getstate(),
        "numpy_legacy": np.random.get_state(),
        "numpy_train": rng_train.bit_generator.state,
        "torch_cpu": torch.get_rng_state(),
    }
    if DEVICE == "cuda":
        state["torch_cuda"] = torch.cuda.get_rng_state_all()
    return state


def set_rng_state(state):
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy_legacy"])
    rng_train.bit_generator.state = state["numpy_train"]
    torch.set_rng_state(state["torch_cpu"])
    if DEVICE == "cuda" and "torch_cuda" in state:
        torch.cuda.set_rng_state_all(state["torch_cuda"])


def save_checkpoint(path, epoch, global_step, history, best_score, best_epoch):
    ckpt = {
        "model": model.state_dict(),
        "optimizer": opt.state_dict(),
        "scheduler": scheduler.state_dict(),
        "epoch": int(epoch),
        "global_step": int(global_step),
        "rng_state": get_rng_state(),
        "norm": dict(norm_payload),
        "best_score": float(best_score),
        "best_epoch": int(best_epoch),
        "history": history,
        "dataset_metadata": dataset_metadata,
        "demo_usage": {
            "demo_usage_count": demo_usage_count.copy(),
            "definition": (
                "counts only exact (operator,trajectory,time) coarse-fine pairs "
                "actually used as in-context demonstrations in successfully "
                "consumed gradient-training batches; query/target and monitor "
                "appearances are excluded"
            ),
            "epoch": int(epoch),
        },
        "config": {
            "RUN_TAG": RUN_TAG,
            "TRAIN_MODE": TRAIN_MODE,
            "DATASET_PATH": str(DATASET_PATH),
            "SEED": SEED,
            "N_OP": N_OP,
            "N_TRAJ": N_TRAJ,
            "N_TIME": N_TIME,
            "NX": NX,
            "GROUP_A_OPS": GROUP_A_OPS.tolist(),
            "GROUP_B_OPS": GROUP_B_OPS.tolist(),
            "BASE_TIME_IDX": BASE_TIME_IDX.tolist(),
            "OOD_TIME_IDX": OOD_TIME_IDX.tolist(),
            "MIXED_TIME_IDX": MIXED_TIME_IDX.tolist(),
            "base_time_range": [float(BASE_TIMES[0]), float(BASE_TIMES[-1])],
            "ood_time_range": [float(OOD_TIMES[0]), float(OOD_TIMES[-1])],
            "group_A_training_pool": (
                "base100_2p505_to_3p000" if TRAIN_MODE == "NO_MIXED"
                else "base100_2p505_to_3p000_plus_ood50_3p005_to_3p250"
            ),
            "group_B_training_pool": "base100_2p505_to_3p000_only",
            "normalization": "all_200_beta_operators_all_20_trajectories_BASE100_2p505_to_3p000_only_matched_between_modes",
            "operator_parameter": "beta",
            "operator_definition": operator_definition,
            "trajectory_identity": "fixed_(beta,u0)",
            "operator_given_explicitly_to_model": False,
            "time_given_explicitly_to_model": True,
            "episode_rule": "same_beta_same_trajectory_M_plus_1_distinct_random_saved_time_slots",
            "demo_time_order": "random_sample_order_not_chronologically_sorted",
            "sequence": "[x,t,uc,uf]*M+[x,tq,ucq]->ufq",
            "MIN_DEMOS": MIN_DEMOS,
            "MAX_DEMOS": MAX_DEMOS,
            "MAX_CTX_ROWS": MAX_CTX_ROWS,
            "PREDICTION_POSITION": "final_valid_query_uc_row_4M_plus_2_zero_based",
            "TRAIN_BATCHING": "mixed_M_independent_per_episode_pad_to_23",
            "BATCH_SIZE": BATCH_SIZE,
            "EPOCHS": EPOCHS,
            "STEPS_PER_EPOCH": STEPS_PER_EPOCH,
            "D_MODEL": D_MODEL,
            "N_HEAD": N_HEAD,
            "N_LAYERS": N_LAYERS,
            "D_FF": D_FF,
            "DROPOUT": DROPOUT,
            "N_PARAMS": n_params,
            "WEIGHT_DECAY": WEIGHT_DECAY,
            "BASE_LR": BASE_LR,
            "MAX_LR": MAX_LR,
            "MIN_LR": MIN_LR,
            "WARMUP_FRAC": WARMUP_FRAC,
            "GRAD_CLIP": GRAD_CLIP,
            "MONITOR": "fixed_512_M5_BASE_only_all_200_operators",
            "checkpoint_selection": "BASE_only_monitor_mean_RelL2",
            "group_B_OOD_training_usage": False,
            "group_B_OOD_checkpoint_usage": False,
            "lf_solver_tag": lf_solver_tag,
            "hf_solver_tag": hf_solver_tag,
            "T": T_saved,
            "dt": dt_saved,
            "base_snapshot_dt": base_snapshot_dt_saved,
            "ood_snapshot_dt": ood_snapshot_dt_saved,
            "demo_usage_tracking": True,
            "demo_usage_tracking_scope": "gradient_training_demonstrations_only_not_queries_or_monitor",
            "demo_usage_identity": "operator_beta_x_trajectory_fixed_u0_x_saved_time",
            "natural_group_A_OOD_fraction": (0.0 if TRAIN_MODE == "NO_MIXED" else 1.0 / 3.0),
        },
    }
    torch.save(ckpt, path)

    # Keep an external copy of the exact BEST demo history as soon as a new
    # best checkpoint is written. This is redundant with ckpt["demo_usage"]
    # but makes later strict evaluation/auditing more convenient.
    if Path(path) == BEST_CKPT:
        np.savez_compressed(
            BEST_DEMO_USAGE_NPZ,
            demo_usage_count=demo_usage_count,
            group_a_ops=GROUP_A_OPS,
            group_b_ops=GROUP_B_OPS,
            base_time_idx=BASE_TIME_IDX,
            ood_time_idx=OOD_TIME_IDX,
            times=times,
            best_epoch=np.asarray([best_epoch], dtype=np.int64),
            definition=np.asarray(
                "gradient-training demonstrations only; queries/targets/monitor excluded"
            ),
        )


def load_ckpt(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)


history = []
best_score = float("inf")
best_epoch = -1
global_step = 0
start_epoch = 1

if LATEST_CKPT.exists():
    print("\nAuto-resume from:", LATEST_CKPT)
    ckpt = load_ckpt(LATEST_CKPT)
    if ckpt.get("config", {}).get("TRAIN_MODE") != TRAIN_MODE:
        raise RuntimeError("Latest checkpoint TRAIN_MODE mismatch")
    model.load_state_dict(ckpt["model"])
    opt.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    global_step = int(ckpt["global_step"])
    last_epoch = int(ckpt["epoch"])
    start_epoch = last_epoch + 1
    history = list(ckpt.get("history", []))
    best_score = float(ckpt.get("best_score", float("inf")))
    best_epoch = int(ckpt.get("best_epoch", -1))
    if last_epoch > 0 and "demo_usage" not in ckpt:
        raise RuntimeError("Cannot strict-resume: latest checkpoint lacks demo_usage")
    if "demo_usage" in ckpt:
        restored_usage = np.asarray(ckpt["demo_usage"]["demo_usage_count"], dtype=np.int64)
        if restored_usage.shape != demo_usage_count.shape:
            raise RuntimeError("demo_usage shape mismatch on resume")
        demo_usage_count[...] = restored_usage
    set_rng_state(ckpt.get("rng_state", {}))
    print(
        f"  resumed after epoch={last_epoch}, next={start_epoch}, "
        f"best={best_score:.6f}% @ {best_epoch}"
    )


# ============================================================
# 14. EPOCH-0 BASE MONITOR FOR FRESH RUN
# ============================================================

if start_epoch == 1:
    mon0 = evaluate_monitor()
    print("\nEpoch 000 BASE monitor:")
    print("  model:", summary_str(mon0["pred"]))
    print("  raw  :", summary_str(mon0["raw"]))

    row0 = {
        "epoch": 0,
        "train_loss_norm_MSE": None,
        "lr": float(opt.param_groups[0]["lr"]),
        "base_monitor_mean_RelL2_pct": mon0["pred"]["mean"],
        "base_monitor_median_RelL2_pct": mon0["pred"]["median"],
        "base_monitor_p90_RelL2_pct": mon0["pred"]["p90"],
        "A_ood_slot_fraction_pct": 0.0,
        "A_ood_query_fraction_pct": 0.0,
    }
    history.append(row0)
    best_score = float(mon0["pred"]["mean"])
    best_epoch = 0
    save_checkpoint(BEST_CKPT, 0, 0, history, best_score, best_epoch)
    save_checkpoint(LATEST_CKPT, 0, 0, history, best_score, best_epoch)


# ============================================================
# 15. TRAIN
# ============================================================

print("\n" + "=" * 130)
print("TRAINING START")
print("=" * 130)

for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()
    model.train()
    losses = []
    m_counts = {m: 0 for m in range(MIN_DEMOS, MAX_DEMOS + 1)}
    audit_epoch = {
        "A_episodes": 0,
        "B_episodes": 0,
        "A_slots": 0,
        "A_ood_slots": 0,
        "A_ood_queries": 0,
        "B_ood_slots": 0,
        "B_ood_queries": 0,
    }

    for _ in range(STEPS_PER_EPOCH):
        (
            m_arr,
            mat_ctx,
            row_types,
            padding_mask,
            query_pos,
            target_n,
            audit,
            track_ops,
            track_traj,
            track_time,
        ) = make_train_batch(BATCH_SIZE)

        for m in m_counts:
            m_counts[m] += int(np.sum(m_arr == m))
        for k in audit_epoch:
            audit_epoch[k] += int(audit[k])

        opt.zero_grad(set_to_none=True)

        pred_all_n = model(
            mat_ctx,
            row_types,
            key_padding_mask=padding_mask,
        )
        batch_ids = torch.arange(BATCH_SIZE, device=DEVICE)
        pred_n = pred_all_n[batch_ids, query_pos, :]

        loss = torch.mean((pred_n - target_n) ** 2)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()

        # STRICT COMMIT: count only demos from a batch whose gradient step
        # actually completed. Query/target appearances are not in track_*.
        np.add.at(
            demo_usage_count,
            (track_ops, track_traj, track_time),
            1,
        )

        scheduler.step()
        global_step += 1
        losses.append(float(loss.detach().cpu()))

    train_loss = float(np.mean(losses))
    lr_now = float(opt.param_groups[0]["lr"])
    sec = float(time.time() - t0)

    a_ood_slot_frac = (
        100.0 * audit_epoch["A_ood_slots"] / max(audit_epoch["A_slots"], 1)
    )
    a_ood_query_frac = (
        100.0 * audit_epoch["A_ood_queries"] / max(audit_epoch["A_episodes"], 1)
    )

    if audit_epoch["B_ood_slots"] != 0 or audit_epoch["B_ood_queries"] != 0:
        raise RuntimeError("Group-B OOD leakage detected")
    if TRAIN_MODE == "NO_MIXED" and audit_epoch["A_ood_slots"] != 0:
        raise RuntimeError("NO_MIXED Group-A OOD leakage detected")

    row = {
        "epoch": int(epoch),
        "train_loss_norm_MSE": train_loss,
        "lr": lr_now,
        "epoch_seconds": sec,
        "A_episodes": audit_epoch["A_episodes"],
        "B_episodes": audit_epoch["B_episodes"],
        "A_ood_slots": audit_epoch["A_ood_slots"],
        "A_ood_queries": audit_epoch["A_ood_queries"],
        "A_ood_slot_fraction_pct": a_ood_slot_frac,
        "A_ood_query_fraction_pct": a_ood_query_frac,
        "B_ood_slots": audit_epoch["B_ood_slots"],
        "B_ood_queries": audit_epoch["B_ood_queries"],
    }

    do_eval = (epoch % PRINT_EVERY == 0) or (epoch == EPOCHS)
    if do_eval:
        mon = evaluate_monitor()
        s = mon["pred"]
        row.update({
            "base_monitor_mean_RelL2_pct": s["mean"],
            "base_monitor_median_RelL2_pct": s["median"],
            "base_monitor_p90_RelL2_pct": s["p90"],
        })

        if s["mean"] < best_score:
            best_score = float(s["mean"])
            best_epoch = int(epoch)
            save_checkpoint(
                BEST_CKPT,
                epoch,
                global_step,
                history + [row],
                best_score,
                best_epoch,
            )

        print("\n" + "-" * 130)
        print(
            f"Epoch {epoch:03d}/{EPOCHS} | "
            f"loss={train_loss:.6e} | lr={lr_now:.3e} | {sec:.1f}s"
        )
        print("  M counts:", m_counts)
        print(
            f"  A/B episodes: {audit_epoch['A_episodes']} / {audit_epoch['B_episodes']}"
        )
        print(
            f"  A OOD slots : {audit_epoch['A_ood_slots']} / {audit_epoch['A_slots']} "
            f"= {a_ood_slot_frac:.3f}%"
        )
        print(
            f"  A OOD query : {audit_epoch['A_ood_queries']} / {audit_epoch['A_episodes']} "
            f"= {a_ood_query_frac:.3f}%"
        )
        print(
            f"  B OOD slots/query: {audit_epoch['B_ood_slots']} / {audit_epoch['B_ood_queries']}"
        )
        usage_now = demo_coverage_summary(demo_usage_count)
        if usage_now["group_B_ood_demo_uses"] != 0:
            raise RuntimeError("STRICT TRACKING LEAKAGE: Group-B OOD demo usage is nonzero")
        print(
            "  Group-B actually-used BASE demo slots/trajectory: "
            f"min={usage_now['group_B_used_base_slots_per_trajectory_min']}, "
            f"mean={usage_now['group_B_used_base_slots_per_trajectory_mean']:.2f}, "
            f"max={usage_now['group_B_used_base_slots_per_trajectory_max']}"
        )
        print("  BASE monitor:", summary_str(s))
        print(f"  BEST BASE monitor: {best_score:.4f}% @ epoch {best_epoch}")

    history.append(row)

    # Latest state every epoch for restart safety.
    save_checkpoint(
        LATEST_CKPT,
        epoch,
        global_step,
        history,
        best_score,
        best_epoch,
    )

    with open(HISTORY_JSON, "w") as f:
        json.dump(history, f, indent=2)
    pd.DataFrame(history).to_csv(HISTORY_CSV, index=False)


# ============================================================
# 16. FINAL CHECKPOINT / AUDIT
# ============================================================

save_checkpoint(
    FINAL_CKPT,
    EPOCHS,
    global_step,
    history,
    best_score,
    best_epoch,
)

best_ckpt = load_ckpt(BEST_CKPT)
if "demo_usage" not in best_ckpt:
    raise RuntimeError("BEST checkpoint is missing strict demo_usage")

best_demo_usage = np.asarray(
    best_ckpt["demo_usage"]["demo_usage_count"],
    dtype=np.int64,
)
final_demo_usage = demo_usage_count.copy()
assert best_demo_usage.shape == (N_OP, N_TRAJ, N_TIME)
assert final_demo_usage.shape == (N_OP, N_TRAJ, N_TIME)

best_usage_summary = demo_coverage_summary(best_demo_usage)
final_usage_summary = demo_coverage_summary(final_demo_usage)

# Absolute leakage guard: Group B can never contribute OOD demonstrations.
assert best_usage_summary["group_B_ood_demo_uses"] == 0
assert final_usage_summary["group_B_ood_demo_uses"] == 0

# Strict K=5 formal evaluation must be possible for EVERY Group-B trajectory.
best_group_b_used_base = np.count_nonzero(
    best_demo_usage[GROUP_B_OPS][:, :, BASE_TIME_IDX] > 0,
    axis=2,
)
assert best_group_b_used_base.shape == (100, 20)
assert np.all(best_group_b_used_base >= MONITOR_FIXED_DEMOS), (
    "At least one Group-B (beta,trajectory) has fewer than five BASE states "
    "actually used as training demonstrations by the selected BEST checkpoint."
)

np.savez_compressed(
    BEST_DEMO_USAGE_NPZ,
    demo_usage_count=best_demo_usage,
    group_a_ops=GROUP_A_OPS,
    group_b_ops=GROUP_B_OPS,
    base_time_idx=BASE_TIME_IDX,
    ood_time_idx=OOD_TIME_IDX,
    times=times,
    best_epoch=np.asarray([best_ckpt["best_epoch"]], dtype=np.int64),
    definition=np.asarray(
        "gradient-training demonstrations only; queries/targets/monitor excluded"
    ),
)
np.savez_compressed(
    FINAL_DEMO_USAGE_NPZ,
    demo_usage_count=final_demo_usage,
    group_a_ops=GROUP_A_OPS,
    group_b_ops=GROUP_B_OPS,
    base_time_idx=BASE_TIME_IDX,
    ood_time_idx=OOD_TIME_IDX,
    times=times,
    final_epoch=np.asarray([EPOCHS], dtype=np.int64),
    definition=np.asarray(
        "gradient-training demonstrations only; queries/targets/monitor excluded"
    ),
)

with open(DEMO_USAGE_AUDIT_JSON, "w") as f:
    json.dump(
        {
            "tracking_definition": (
                "exact (beta operator, fixed trajectory/IC, saved time) states "
                "used as in-context demonstrations in completed gradient steps only; "
                "queries/targets/monitor excluded"
            ),
            "best_epoch": int(best_ckpt["best_epoch"]),
            "best_checkpoint": best_usage_summary,
            "final_training_state": final_usage_summary,
            "strict_K5_groupB_all_trajectories_valid": bool(
                np.all(best_group_b_used_base >= MONITOR_FIXED_DEMOS)
            ),
        },
        f,
        indent=2,
    )

model.load_state_dict(best_ckpt["model"])
final_mon = evaluate_monitor()

print("\n" + "=" * 130)
print("TRAINING COMPLETE")
print("=" * 130)
print("MODE          :", TRAIN_MODE)
print("BEST epoch    :", best_ckpt["best_epoch"])
print("BEST score    :", f"{best_ckpt['best_score']:.6f}%")
print("BASE monitor  :", summary_str(final_mon["pred"]))
print("BEST checkpoint:", BEST_CKPT)
print("FINAL checkpoint:", FINAL_CKPT)
print("LATEST checkpoint:", LATEST_CKPT)
print("History CSV   :", HISTORY_CSV)
print("Group split   :", SAVE_DIR / "crossop_group_split.npz")
print("Monitor cases :", SAVE_DIR / "fixed_base_monitor_cases.csv")
print("BEST demo usage:", BEST_DEMO_USAGE_NPZ)
print("FINAL demo usage:", FINAL_DEMO_USAGE_NPZ)
print("Demo audit JSON :", DEMO_USAGE_AUDIT_JSON)
print(
    "BEST Group-B used BASE demo slots/trajectory: ",
    f"min={best_usage_summary['group_B_used_base_slots_per_trajectory_min']}, ",
    f"mean={best_usage_summary['group_B_used_base_slots_per_trajectory_mean']:.2f}, ",
    f"max={best_usage_summary['group_B_used_base_slots_per_trajectory_max']}"
)
print("BEST Group-B OOD demo uses:", best_usage_summary["group_B_ood_demo_uses"])
print("=" * 130)

KS CROSS-OPERATOR TEMPORAL TRAINING -- NO_MIXED
DEVICE       : cuda
DATASET_ROOT : /kaggle/input/notebooks/xinyuanli913/ks-dataset
SAVE_DIR     : /kaggle/working/ks_beta20_latebase_B100_OOD50_T3p25_nomixed_Nop200_Ntraj20_A100_B100_strictDemo_large288_H6_L5_FF576_mixedM_pad23_150ep900step
Group split  : fixed A100/B100 with seed=42
BASE100      : 2.505..3.000, spacing=0.005
OOD50        : 3.005..3.250, spacing=0.005
Train pools  : A=BASE, B=BASE
Normalization: matched BASE100-only across all 200 beta x 20 trajectories
Monitor      : fixed 512 BASE100-only M=5 cases; selects BEST
B-OOD usage  : NONE inside training
Demo tracking: STRICT gradient-training demos only; queries/monitor excluded

DATASET_PATH: /kaggle/input/notebooks/xinyuanli913/ks-dataset/ks_mf_queryt_beta070_085_FD2LF72_SpecHF256_T3p25_B100late_OOD50_dt001_Nop200_Ntraj20_Nt150_Nr100/ks_mf_queryt_beta070_085_FD2LF72_SpecHF256_T3p25_B100late_OOD50_dt001_Nop200_Ntraj20_Nt150_Nr100.npz

Dataset audit:
  beta        : (200,) [0

/tmp/ipykernel_23/1779433136.py:851: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(



Model:
  d_model / heads / layers / FF / dropout: 288 6 5 576 0.1
  trainable parameters: 3,484,036

Padding sanity max diff: 0.000e+00

Optimizer:
  total steps  : 135000
  warmup steps : 10800
  LR           : 0.0001 -> 0.0002 -> 1e-05

Epoch 000 BASE monitor:
  model: mean=102.8114% | median=102.6834% | std=2.1205% | p90=105.4142% | max=109.3399%
  raw  : mean=23.5254% | median=12.9667% | std=31.3765% | p90=52.6599% | max=202.5493%

TRAINING START

----------------------------------------------------------------------------------------------------------------------------------
Epoch 005/150 | loss=1.624832e-02 | lr=1.417e-04 | 264.9s
  M counts: {1: 179500, 2: 179803, 3: 180366, 4: 179864, 5: 180467}
  A/B episodes: 450180 / 449820
  A OOD slots : 0 / 1800914 = 0.000%
  A OOD query : 0 / 450180 = 0.000%
  B OOD slots/query: 0 / 0
  Group-B actually-used BASE demo slots/trajectory: min=100, mean=100.00, max=100
  BASE monitor: mean=6.2282% | median=4.8039% | std=5.0043% | p90=10.246